### This ARTICLES notebook 

-- Loads journal data into the database  
-- Finds the ISSN of Domingo's Incites journals  
-- EXamines the "completeness" of the Incites journals 


In [71]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [72]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=4_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [ ]:
class ArticlesETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_journals(self):
        # Extract inCites-OpenAlex journal table (ISSN and OA journal_id)
        self.journals = self.db.sql("SELECT * FROM sources_oa_incites").df().rename(columns={'id': 'source_id'}).sort_values('works_count')
        print(f'{self.journals.shape = }\n{self.journals.head()}')
        return
    
    def extract_works_by_journal(self):
        # Extract OA wirks for the journal set, for publication years 2010+ to now
        hold = []
        for row in self.journals.itertuples():
            source_id = row.source_id
            reader = rf'Works().filter(primary_location={{"source": {{"id": "{source_id}"}}}}).filter(publication_year=">2009")'
            if oa := self.read_openalex(reader=reader):
                print(f'EXTRACTED {len(oa) = } WORKS FOR {row.display_name = }')
                hold.extend(oa)
            else:
                print(f'OpenAlex does not have articles for {source_id = } {row.display_name = }')
        df = pd.DataFrame.from_records(hold)
        print(f'{df.shape = }\n{df.head()}')        
        return
    
    def read_openalex(self, reader=None):
        # using the pyalex API call "reader" as key, run the API call, cache the response, and return it (JSON) 
        if result := self.cache.get(reader):
            return result
        try:
            result = list(chain(*eval(reader).paginate(per_page=200)))
        except Exception as e:
            print(f'FAILED TO READ cache or OpenAlex API with {reader = }')
            print(f'{e = }')
            return
        self.cache[reader] = result
        return result


In [74]:
def main():

    jetl = ArticlesETL()
    jetl.extract_journals()
    jetl.extract_works_by_journal()
    # jetl.match_incites_oa() 

In [75]:
if __name__ == "__main__":
    main()
    print("DONE!")


┌──────────┬─────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────